# 1. Setup

### 1.1 Install deps & packages

In [ ]:
%pip install numpy pandas matplotlib kagglehub
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import kagglehub
import shutil
import os

### 1.2 Download dataset

In [ ]:
try:
    os.mkdir('original_dataset')
    kagglehub.dataset_download("zahranusratt/banking-fraud-detection-dataset", output_dir='original_dataset')
except FileExistsError:
    pass

# Remove uesless metadata
shutil.rmtree('original_dataset/.complete/', ignore_errors=True)

print('Dataset downloaded')

# 2. Data pipeline

### 2.1 Load csv into dataframe

In [ ]:
fraud_data = pd.read_csv('original_dataset/FraudShield_Banking_Data (1).csv')

print(f"Shape\n{fraud_data.shape}\n")
print(f"Head\n{fraud_data.head()}")
print("\nColumns")
for column_number, column_name in enumerate(fraud_data.columns, start=1):
    print(column_number, column_name)

In [ ]:
fraud_data_original = fraud_data.copy(deep=True)

print("Records:", fraud_data_original.shape[0])
print("Columns:", fraud_data_original.shape[1])

print("\nData-type counts:")
print(fraud_data_original.dtypes.value_counts())

In [ ]:
summary = pd.DataFrame({
    'dtype': fraud_data_original.dtypes,
    'sample_values': [fraud_data_original[col].dropna().unique()[:3] for col in fraud_data_original.columns]
})
print(summary.to_string())

2.  Numeric ranges, missing values, duplicates, and inconsistent entries

In [ ]:


print("\n" + "="*60)
print("NUMERIC RANGES")
print("="*60)
print(fraud_data_original.describe())

print("\n" + "="*60)
print("CATEGORICAL SUMMARY")
print("="*60)
print(fraud_data_original.describe(include='object'))

print("\n" + "="*60)
print("UNIQUE VALUES (categorical columns)")
print("="*60)
for col in fraud_data_original.select_dtypes(include="object").columns:
    print(f"\n{col}: {fraud_data_original[col].nunique()} unique values")
    print(fraud_data_original[col].unique()[:15])

print("\n" + "="*60)
print("MISSING VALUES")
print("="*60)
missing = fraud_data_original.isna().sum()
missing_pct = (missing / len(fraud_data_original) * 100).round(2)
missing_summary = pd.DataFrame({"missing": missing, "pct": missing_pct})
print(missing_summary[missing_summary["missing"] > 0].sort_values("missing", ascending=False))

print("\n" + "="*60)
print("DUPLICATE ROWS")
print("="*60)
print("Full duplicate rows:", fraud_data_original.duplicated().sum())
if "transaction_id" in fraud_data_original.columns:
    print("Duplicate transaction_id:", fraud_data_original["transaction_id"].duplicated().sum())

## 3.1 Dropping columns

Looking at the dataset, we can see the transaction ID is unique to every row, therefore it won't be useful in deriving any value. We can use the customer contry to identify if the transaction took place in a high risk country, for this reason we can drop the city as it's too spesific.

In [ ]:
fraud_data = fraud_data.drop(columns=['Transaction_ID', 'IP_Address'])

fraud_data.head()

### 3.2 Normalising data

Since a lot of these columns are categories we can convert them to a number using a dict

In [ ]:
categorical_columns = ['Merchant_Category', 'Transaction_Type', 'Card_Type', 'Is_International_Transaction', 'Is_New_Merchant', 'Unusual_Time_Transaction']

category_values = {}

for col in categorical_columns:
    categories = fraud_data[col].astype('category').cat.categories.tolist()
    category_values[col] = categories
    fraud_data[col] = fraud_data[col].astype('category').cat.codes

# Show the mapping
for col, cats in category_values.items():
    print(col, dict(enumerate(cats)))

fraud_data.head()

### 3.3 Flagging potentially risky transactions

A transaction is flagged as `potentially_risky` when 2 or more of the following signals are present:
- Is an international transaction
- Is with a new merchant
- Occurs at an unusual time
- Has a failed transaction count > 0
- Has a previous fraud history
- Distance from home > 1000
- Transaction amount ≥ 10 million

In [ ]:
# Convert fraud labels to numbers while keeping missing labels
fraud_data['Fraud_Label'] = fraud_data['Fraud_Label'].map({
    'Normal': 0,
    'Fraud': 1
})

print(fraud_data['Fraud_Label'].value_counts(dropna=False))

In [ ]:
risk_conditions = (
    (fraud_data['Is_International_Transaction'] == 1).astype(int) +
    (fraud_data['Is_New_Merchant'] == 1).astype(int) +
    (fraud_data['Unusual_Time_Transaction'] == 1).astype(int) +
    (fraud_data['Failed_Transaction_Count'] > 0).astype(int) +
    (fraud_data['Previous_Fraud_Count'] > 0).astype(int) +
    (fraud_data['Distance_From_Home'] > 1000).astype(int) +
    (fraud_data['Transaction_Amount (in Million)'] >= 10).astype(int)
)

fraud_data['potentially_risky'] = (risk_conditions >= 2).astype(int)

risky_count = fraud_data['potentially_risky'].sum()
print(f"Flagged {risky_count} potentially risky transactions ({risky_count / len(fraud_data) * 100:.1f}%)")
print(f"\nFraud rate in risky transactions:     {fraud_data[fraud_data['potentially_risky'] == 1]['Fraud_Label'].mean():.3f}")
print(f"Fraud rate in non-risky transactions: {fraud_data[fraud_data['potentially_risky'] == 0]['Fraud_Label'].mean():.3f}")
fraud_data['potentially_risky'].value_counts()

## 4. Exploratory Data Analysis

### 4.1 Basic Dataset Overview


In [ ]:
# Check missing categories across the columns encoded in Task 3
encoded_cols = list(category_values.keys())
encoded_missing = (fraud_data[encoded_cols] == -1).sum()

print(encoded_missing)
print("Total missing category entries:", encoded_missing.sum())

In [ ]:
# Shows only columns with missing values

missing = fraud_data.isna().sum()
print(missing[missing > 0].sort_values(ascending=False))

In [ ]:
# Checks for missing categories stored as -1 after encoding

encoded_cols = [
    'Merchant_Category',
    'Transaction_Type',
    'Card_Type',
    'Transaction_Location',
    'Customer_Home_Location'
]

print((fraud_data[encoded_cols] == -1).sum())

After completing Task 3, we checked the dataset and found that it contains 49,994 records and 25 columns. We noticed that missing values still remain in 18 columns, including four missing fraud labels. Failed_Transaction_Count has the most, with 11 missing values. We also found 26 missing entries stored as -1 across five encoded columns, which were not included in the isna() counts. These represent missing information, not actual categories. When calculating fraud percentages, we excluded the four missing labels so they were not counted as normal transactions.


### 4.2 Descriptive Analysis

In [ ]:
# Summarise numerical features to understand their values and spread

summary_cols = [
    'Transaction_Amount (in Million)',
    'Account_Balance (in Million)',
    'Distance_From_Home',
    'Daily_Transaction_Count',
    'Weekly_Transaction_Count',
    'Failed_Transaction_Count',
    'Previous_Fraud_Count'
]

fraud_data[summary_cols].describe().T

In [ ]:
# Count transactions in each category, including missing-category codes

categorical_cols = [
    'Transaction_Type',
    'Merchant_Category',
    'Card_Type'
]

for col in categorical_cols:
    print("\n", col)
    print("Category names:", dict(enumerate(category_values[col])))
    print("Missing category code: -1")
    print(fraud_data[col].value_counts().sort_index())

In [ ]:
# Display the names linked to the card-type codes

print(dict(enumerate(category_values['Card_Type'])))

We summarised relevant numerical features to understand their typical values and spread. Transaction amounts range from 1 to 9 million, with a mean and median of approximately 5 million. Daily transaction counts average around 4, while weekly counts average around 12.52. The counts differ between columns because missing values are excluded from these calculations.

We also checked transaction types, merchant categories and card types. ATM, Online and POS transactions have similar counts, and the six merchant categories are fairly evenly represented. Credit cards account for 24,887 transactions and debit cards account for 25,104, with three missing card types. Although these categories are fairly balanced, this does not mean normal and fraudulent transactions are equally represented.


### 4.3 Fraud Class Distribution

In [ ]:
# Count normal and fraudulent transactions, excluding missing labels

print("Transaction counts (0 = Normal, 1 = Fraud):")
print(fraud_data['Fraud_Label'].value_counts().sort_index())

print("\nPercentage of transactions with a known label:")
print(
    fraud_data['Fraud_Label']
    .value_counts(normalize=True)
    .sort_index() * 100
)

print("\nMissing fraud labels:")
print(fraud_data['Fraud_Label'].isna().sum())

We found that 47,567 transactions were labelled normal and 2,423 were labelled fraudulent. Of the 49,990 transactions with known labels, 95.15% were normal and 4.85% were fraudulent. The four missing labels were excluded from these percentages. This shows a clear class imbalance, with normal transactions making up most of the dataset. When evaluating a model later, accuracy alone could be misleading because predicting every transaction as normal would already give approximately 95.15% accuracy while missing all fraudulent transactions.


### 4.4 Data Visualisations

In [ ]:
# Compare the number of normal and fraudulent transactions

fraud_counts = fraud_data['Fraud_Label'].value_counts().sort_index()

plt.bar(['Normal', 'Fraud'], fraud_counts.values)
plt.xlabel('Transaction class')
plt.ylabel('Number of transactions')
plt.title('Normal and Fraudulent Transactions')
plt.show()

In [ ]:
# Show the distribution of transaction amounts

plt.hist(
    fraud_data['Transaction_Amount (in Million)'].dropna(),
    bins=9,
    edgecolor='black'
)
plt.xlabel('Transaction amount (in millions)')
plt.ylabel('Number of transactions')
plt.title('Distribution of Transaction Amounts')
plt.show()

In [ ]:
# Show the median, spread and potential outliers in transaction amounts

plt.boxplot(
    fraud_data['Transaction_Amount (in Million)'].dropna()
)
plt.xticks([1], ['Transaction amount'])
plt.ylabel('Transaction amount (in millions)')
plt.title('Boxplot of Transaction Amounts')
plt.show()

The bar chart shows that normal transactions greatly outnumber fraudulent transactions. From the histogram, we can see that transaction amounts are roughly evenly distributed between 1 and 9 million. The boxplot shows a median of 5 million, with the middle 50% of values between 3 and 7 million. No transaction amounts were flagged as outliers using the default 1.5×IQR rule. However, this does not mean there are no fraudulent transactions, as an unusual amount alone does not determine whether a transaction is fraud.


### 4.5 Relationships and Unusual Patterns

In [ ]:
# Compare fraud rates across transaction amounts

amount_fraud_rate = (
    fraud_data.groupby('Transaction_Amount (in Million)')['Fraud_Label']
    .mean() * 100
)

print(amount_fraud_rate)

plt.bar(amount_fraud_rate.index, amount_fraud_rate.values)
plt.xlabel('Transaction amount (in millions)')
plt.ylabel('Fraud rate (%)')
plt.title('Fraud Rate by Transaction Amount')
plt.show()